# Trossen OSS — Catalog Welcome

The interactive companion to `pixi run query`. It connects to the **local** Rerun
catalog, embeds the Viewer inline, and runs the same cross-episode questions
that `trossen_oss.query` answers headless.

## Build the catalog first

In separate shells, from the repo root:

```bash
pixi run preprocess        # MCAP episodes -> outputs/rrds/*.rrd  (+ blueprint)
pixi run serve             # start the local catalog on :51234  (leave running)
pixi run register          # register the RRDs as the `trossen_oss` dataset
```

`preprocess` converts **every** episode by default; pass `--num-rrd-to-process 10`
for a quick subset.

## Connect

In [ ]:
from __future__ import annotations

import rerun as rr
from rerun.notebook import Viewer

from trossen_oss.catalog import DEFAULT_CATALOG_URL

dataset_name = "trossen_oss"

url = DEFAULT_CATALOG_URL
client = rr.catalog.CatalogClient(DEFAULT_CATALOG_URL)
dataset = client.get_dataset(dataset_name)
dataset.name

'trossen_oss'

## View the episodes

Browse the catalog and open a segment. The dataset's default blueprint (camera
grid, 3D scene, and per-joint signal graphs) loads automatically.

In [4]:
Viewer(width=1200, height=640, url=url)

HTML(value='<div id="3940a258-fb32-40e2-b5e7-99f83fe306e7"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

## Inventory the dataset

One cheap `segment_table()` round-trip: one row per episode (segment id, layers,
chunk count, byte size).

In [ ]:
from trossen_oss.query import segment_overview

segment_overview(dataset)

## Which arm is scripted vs. task-driven?

Summed joint travel (`max − min`) per arm, per episode. One arm follows a fixed
script (≈constant across episodes); the other is task-driven (varies).

In [ ]:
from trossen_oss.query import arm_activity

arm_activity(client, dataset)

## How often does a joint exceed its URDF velocity limit?

Finite-difference the position signal and count samples above the joint's
`π rad/s` revolute limit, per episode (peak velocity descending).

In [ ]:
from trossen_oss.query import velocity_limit_violations

velocity_limit_violations(client, dataset, "/robot_right/joints/joint_1")